In [ ]:
if ort_session =ort.InferenceSession('./model/best_lgb_model_260411.onnx')
with open(output_txt,'w',encoding='utf-8') as f:
    for ind, file_path in enumerate(csv_list1):
        try:
            data=normalize_data(file_path)
            data=process_folder(data)
            final_pred=0
            X=pd.DataFrame(data).values
            X=scaler.transform(X).astype(np.float32)
            if np.isnan(X).any():
                raise ValueError("数据包含NaN")
            onnx_output=ort_session.run(None,{"input":X})[0]
            outputs =torch.from_numpy(onnx_output)
            y_pred_prob=torch.softmax(outputs,dim=1)
            score=y_pred_prob[0][1]
            final_pred=int(score>THRES)
            if '/20260319_abnormal_data/' in file_path:
                label=1
            else:
                label=0
            if label=0:
                if final_pred !=0:
                    metric["fp"]+=1
                else:
                    metric["tn"]+=1
            else:
                if final_pred !=0:
                    metric["tp"]+=1
                else:
                    metric["fn"]+=1
            file_count+=1
            f.write(f"{file_path},{score},{final_pred},{label}\n")
        except:
            print(file_path)
            countinue
print(time.time()-start)
print((time.time()-start)/file_count)
print(metric)
score=